# Week 5 Exercise — Optimized RAG Pipeline for Insurellm

**By:** Jaymineh  
**API:** OpenRouter (gpt-4o-mini for generation + query rewriting)

An optimized RAG pipeline that beats the baseline evaluation scores by upgrading the embedding model,
adding hierarchical summaries, and simplifying the retrieval pipeline.

**Week 5 skills demonstrated:**
- Building and evaluating a **RAG pipeline** with retrieval metrics (MRR, nDCG, coverage) and LLM-as-judge
- **Hierarchical RAG** with LLM-generated per-document and per-category summaries
- **Embedding model selection** — benchmarking all-MiniLM-L6-v2 vs BAAI/bge-base-en-v1.5
- **Query rewriting** for improved vector similarity search
- **MMR retrieval** for diverse document retrieval across spanning/relationship questions
- Iterative improvement: measuring, changing one variable, re-measuring

## Score Progression

| Metric | Baseline | v1 (MiniLM + reranking) | v2 (BGE + Gemini) | v3 Final (BGE + gpt-4o-mini) |
|--------|----------|------------------------|--------------------|-------------------------------|
| MRR | 0.7613 | 0.7910 | 0.8540 | **0.8506** |
| nDCG | 0.7670 | 0.7926 | 0.8459 | **0.8432** |
| Coverage | 92.1% | 95.5% | 96.9% | **97.1%** |
| Accuracy | 4.20/5 | 4.27/5 | 4.38/5 | **4.45/5** |
| Completeness | 3.85/5 | 3.98/5 | 3.89/5 | **4.07/5** |
| Relevance | 4.49/5 | 4.54/5 | 4.45/5 | **4.61/5** |

## Key Finding: Model Alignment Matters
Gemini 2.5 Flash produced more accurate answers (4.38 accuracy) but scored lower on
completeness (3.89) and relevance (4.45) when judged by gpt-4o-mini.
Using gpt-4o-mini for both generation and judging yielded the best overall scores.

## Key Files
- `implementation/ingest.py` — Hierarchical RAG ingestion with BGE embeddings
- `implementation/answer.py` — Query rewriting, MMR retrieval, gpt-4o-mini answer generation

## Setup

In [ ]:
import os
import sys
import inspect
from dotenv import load_dotenv

os.chdir(os.path.join(os.path.dirname(os.getcwd()), '..'))

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set")

## Part 1: Embedding Model Upgrade

The single highest-impact change: replacing `all-MiniLM-L6-v2` (56.3 MTEB, 384 dims, 256-token context)
with `BAAI/bge-base-en-v1.5` (~63 MTEB, 768 dims, 512-token context).

This improves retrieval quality across the board:
- Higher MTEB score means better semantic matching
- 768 dimensions capture finer distinctions between concepts
- 512-token context avoids silent truncation of longer chunks

### Why BGE-base over alternatives?
- **vs nomic-embed-text-v1.5** (59.4 MTEB): Lower quality despite being larger. Requires task prefix system.
- **vs BAAI/bge-m3** (63.0 MTEB): 5x model size for same quality. Designed for multilingual (not needed).
- **vs BAAI/bge-large-en-v1.5**: 3x model size for ~1 MTEB point. Diminishing returns.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(persist_directory="vector_db", embedding_function=embeddings)
collection = vectorstore._collection

count = collection.count()
sample = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
print(f"Vector store: {count} vectors with {len(sample)} dimensions")
print(f"Embedding model: BAAI/bge-base-en-v1.5 (768d, ~63 MTEB)")
print(f"Previous: all-MiniLM-L6-v2 (384d, 56.3 MTEB)")

## Part 2: Hierarchical RAG (Ingestion)

Alongside standard 800-char chunks, the ingestion pipeline generates LLM-powered summaries:

1. **Per-document summaries** (76 docs) — comprehensive summaries capturing all key facts
2. **Per-category summaries** (4 categories) — overview of employees, products, contracts, company

This directly targets **holistic questions** (the weakest baseline category at ~1.5/5 accuracy).

In [ ]:
from implementation import ingest

print("=== Ingestion Configuration ===")
print(f"Embedding: BAAI/bge-base-en-v1.5")
print(f"Chunk size: 800 chars, 200 overlap")
print(f"Summaries: per-document (76) + per-category (4)")
print(f"LLM for summaries: gpt-4o-mini via OpenRouter")
print()
print("=== generate_document_summary ===")
print(inspect.getsource(ingest.generate_document_summary))

## Part 3: Optimized Answer Pipeline

### Architecture: 2 LLM calls per question (down from 3)

```
query --> rewrite (gpt-4o-mini) --> retrieve 15 docs (BGE + MMR) --> answer (gpt-4o-mini)
```

### Key design decisions:

**Removed LLM reranking**: The v1 reranker only saw 200 chars per chunk — too little for good decisions.
Fixing it (full text) would mean ~15K tokens per reranking call. With better embeddings (BGE),
the raw retrieval order is good enough. Removing reranking saves 1 LLM call per question = ~33% faster.

**Same model for generation and judging**: We tested Gemini 2.5 Flash for answer generation — it
improved accuracy (4.38) but hurt completeness (3.89) and relevance (4.45) due to style mismatch
with the gpt-4o-mini judge. Using gpt-4o-mini for everything produced the best overall scores.

**MMR with K=15**: Maximum Marginal Relevance provides diverse results for spanning/relationship
questions. K=15 ensures nDCG's top-10 window is well-filled, plus 5 extras for coverage.

In [ ]:
from implementation import answer

print(f"Answer model: {answer.ANSWER_MODEL}")
print(f"Utility model: {answer.UTILITY_MODEL}")
print(f"Retrieval K: {answer.RETRIEVAL_K}")
print()
print("=== rewrite_query ===")
print(inspect.getsource(answer.rewrite_query))
print("\n=== fetch_context ===")
print(inspect.getsource(answer.fetch_context))

In [ ]:
# Test: Query rewriting
from implementation.answer import rewrite_query

test_questions = [
    "Who won the IIOTY award?",
    "What products does Insurellm offer?",
    "Who went to Manchester University?",
]

for q in test_questions:
    rewritten = rewrite_query(q)
    print(f"Original:  {q}")
    print(f"Rewritten: {rewritten}")
    print()

In [ ]:
# Test: Full answer pipeline with Gemini 2.5 Flash
from implementation.answer import answer_question

question = "Who won the IIOTY award in 2023?"
answer_text, docs = answer_question(question)
print(f"Q: {question}")
print(f"A: {answer_text}")
print(f"\nRetrieved {len(docs)} context documents")

In [ ]:
# Test a holistic question (weakest baseline category)
question = "What is Insurellm and what does the company do?"
answer_text, docs = answer_question(question)
print(f"Q: {question}")
print(f"A: {answer_text}")

## Part 4: Evaluation

Run the full evaluation dashboard:
```bash
uv run evaluator.py
```

The evaluation is faster than v1 (2 LLM calls per question instead of 3).

In [ ]:
# Quick sample evaluation (first 5 tests)
from evaluation.eval import evaluate_retrieval, evaluate_answer
from evaluation.test import load_tests

tests = load_tests()
print(f"Loaded {len(tests)} tests")

print("\n--- Retrieval Sample (first 5 tests) ---")
for test in tests[:5]:
    result = evaluate_retrieval(test)
    print(f"  Q: {test.question[:60]}...")
    print(f"    MRR={result.mrr:.3f}  nDCG={result.ndcg:.3f}  Coverage={result.keyword_coverage:.1f}%")

In [ ]:
# Answer evaluation sample (first 5 tests)
print("--- Answer Sample (first 5 tests) ---")
for test in tests[:5]:
    eval_result, gen_answer, _ = evaluate_answer(test)
    print(f"  Q: {test.question[:60]}...")
    print(f"    Accuracy={eval_result.accuracy:.1f}  Completeness={eval_result.completeness:.1f}  Relevance={eval_result.relevance:.1f}")
    print(f"    Feedback: {eval_result.feedback[:120]}...")
    print()

## Summary of Changes

| Component | Baseline | v1 | v2 | v3 (Final) |
|-----------|----------|-----|-----|------------|
| Embedding model | all-MiniLM-L6-v2 (56 MTEB) | all-MiniLM-L6-v2 | BAAI/bge-base-en-v1.5 | **BAAI/bge-base-en-v1.5 (~63 MTEB)** |
| Embedding dims | 384 | 384 | 768 | **768** |
| Chunk size | 500 | 800 | 800 | 800 |
| Summaries | None | 76 doc + 4 category | 76 doc + 4 category | 76 doc + 4 category |
| Total vectors | 970 | 998 | 1,003 | **1,003** |
| Query rewriting | No | Yes (gpt-4o-mini) | Yes (gpt-4o-mini) | Yes (gpt-4o-mini) |
| Retrieval | Similarity, K=10 | MMR, K=25, fetch_k=50 | MMR, K=15, fetch_k=40 | **MMR, K=15, fetch_k=40** |
| Reranking | No | LLM (200 chars) | Removed | **Removed** |
| Answer LLM | gpt-4o-mini | gpt-4o-mini | Gemini 2.5 Flash | **gpt-4o-mini** |
| LLM calls/question | 1 | 3 | 2 | **2** |
| LLM backend | OpenAI via LangChain | OpenRouter via litellm | OpenRouter via litellm | OpenRouter via litellm |

## Lessons Learned

1. **Embedding quality is the single biggest lever** — upgrading from MiniLM to BGE-base improved retrieval metrics more than query rewriting and LLM reranking combined.
2. **LLM reranking has diminishing returns** — showing only 200 chars per chunk made reranking nearly useless. Better embeddings eliminated the need for it entirely.
3. **Model alignment matters for LLM-as-judge** — Gemini 2.5 Flash gave more accurate answers but scored worse because the gpt-4o-mini judge penalized its verbose style. Using the same model for generation and judging yielded the best scores.
4. **Simplicity wins** — the final pipeline uses fewer LLM calls (2 vs 3) and runs faster while scoring better.